# Transcribe Pipeline
Run in Google Colab: mount Drive, use Whisper large-v3 and Pyannote speaker diarization.


In [ ]:
!pip -q install -U faster-whisper pyannote.audio==3.1.1 torch torchaudio
from google.colab import drive, userdata
from pathlib import Path
import json, torch
drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/transcribe'); INPUT=ROOT/'input'; OUTPUT=ROOT/'output'; STATE=ROOT/'state'
for p in (INPUT,OUTPUT,STATE): p.mkdir(parents=True,exist_ok=True)
assert torch.cuda.is_available(), 'Enable a Colab GPU runtime.'
HF_TOKEN=userdata.get('HUGGINGFACE_TOKEN')
from faster_whisper import WhisperModel
from pyannote.audio import Pipeline
whisper=WhisperModel('large-v3',device='cuda',compute_type='float16')
diarizer=Pipeline.from_pretrained('pyannote/speaker-diarization-3.1',use_auth_token=HF_TOKEN)
diarizer.to(torch.device('cuda'))
registry=STATE/'processed.json'; processed=json.loads(registry.read_text()) if registry.exists() else {}
audio_ext={'.wav','.mp3','.m4a','.flac','.ogg','.opus','.aac','.webm'}
def stamp(x):
 x=int(max(0,x)); return f'{x//3600:02d}:{x%3600//60:02d}:{x%60:02d}'
def assign(a,b,turns):
 scores={}
 for x,y,s in turns: scores[s]=scores.get(s,0)+max(0,min(b,y)-max(a,x))
 return max(scores,key=scores.get) if scores else 'UNKNOWN'
for audio in sorted(INPUT.rglob('*')):
 key=str(audio.relative_to(INPUT))
 if not audio.is_file() or audio.suffix.lower() not in audio_ext or key in processed: continue
 turns=[(s.start,s.end,sp) for s,_,sp in diarizer(str(audio)).itertracks(yield_label=True)]
 segments,_=whisper.transcribe(str(audio),beam_size=5,vad_filter=True); rows=[]
 for s in segments:
  if s.text.strip(): rows.append({'start':s.start,'end':s.end,'speaker':assign(s.start,s.end,turns),'text':s.text.strip()})
 target=(OUTPUT/key).with_suffix(''); target.parent.mkdir(parents=True,exist_ok=True)
 target.with_suffix('.txt').write_text('\n'.join(f'[{stamp(r["start"])} - {r["speaker"]}]: {r["text"]}' for r in rows)+'\n',encoding='utf-8')
 target.with_suffix('.json').write_text(json.dumps({'source':key,'model':'openai/whisper-large-v3','diarization':'pyannote/speaker-diarization-3.1','segments':rows},ensure_ascii=False,indent=2),encoding='utf-8')
 processed[key]=True; registry.write_text(json.dumps(processed,indent=2),encoding='utf-8'); print('DONE',key)


Accept the Pyannote model terms on Hugging Face and create the Colab Secret HUGGINGFACE_TOKEN. Free Colab sessions can disconnect; no browser snippet guarantees persistence.
